# Phase 1 **EXTRACT**

In [2]:
import sqlite3
import pandas as pd

# 1. โหลดข้อมูลดิบจากไฟล์ CSV
df_raw = pd.read_csv('raw_ecommerce_data.csv')

# 2. ตรวจสอบโครงสร้างและค่า Null
print("=== โครงสร้างข้อมูลดิบ (Data Types & Missing Values) ===")
df_raw.info()

print("\n=== ตัวอย่างข้อมูลดิบ 5 แถวแรก ===")
df_raw.head()

=== โครงสร้างข้อมูลดิบ (Data Types & Missing Values) ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185 entries, 0 to 184
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Order_ID       185 non-null    object
 1   Customer_Name  183 non-null    object
 2   Email          184 non-null    object
 3   Product        185 non-null    object
 4   Category       184 non-null    object
 5   Order_Date     185 non-null    object
 6   Quantity       185 non-null    int64 
 7   Unit_Price     185 non-null    object
 8   Amount         142 non-null    object
dtypes: int64(1), object(8)
memory usage: 13.1+ KB

=== ตัวอย่างข้อมูลดิบ 5 แถวแรก ===


,Order_ID,Customer_Name,Email,Product,Category,Order_Date,Quantity,Unit_Price,Amount
0,ORD-0036,Emma Brown,emma.brown@email.com,Monitor 24 inch,Electronics,24/04/2026,1,4131.00,"4,131.00"
1,ORD-0076,linda park,linda.park@email.com,Mechanical Keyboard,Electronics,"Apr 03, 2026",5,1669.50,8347.50
2,ORD-0101,john doe,john@email.com,Wireless Mouse,Electronics,11/04/2026,3,405.00,NaN
3,ORD-0059,Jane Smith,jane@email.com,Wireless Mouse,ELECTRONICS,13/03/2026,2,405.00,฿810.00
4,ORD-0038,PETER KIM,peter.kim@email.com,Gel Pen Set,Stationery,2026-02-28,5,85.50,427.50


# **Data Cleaning**

In [3]:
# คัดลอกข้อมูลมาทำความสะอาด
df_clean = df_raw.copy()

# ทำความสะอาดข้อมูลข้อความ (ตัดช่องว่าง / ปรับตัวพิมพ์เล็ก-ใหญ่)
df_clean['Customer_Name'] = (
    df_clean['Customer_Name']
    .fillna('Unknown')
    .astype(str)
    .str.strip()
    .str.title()
)
df_clean['Email'] = (
    df_clean['Email'].fillna('unknown@email.com').astype(str).str.strip().str.lower()
)
df_clean['Product'] = df_clean['Product'].astype(str).str.strip()
df_clean['Category'] = (
    df_clean['Category'].fillna('Uncategorized').astype(str).str.strip().str.title()
)

# ลบเครื่องหมาย ฿ และ Comma ออกจากตัวเลข
df_clean['Unit_Price'] = (
    df_clean['Unit_Price']
    .astype(str)
    .str.replace(',', '')
    .str.replace('฿', '')
    .str.strip()
    .astype(float)
)
df_clean['Amount'] = (
    df_clean['Amount']
    .astype(str)
    .str.replace(',', '')
    .str.replace('฿', '')
    .str.strip()
)
df_clean['Amount'] = pd.to_numeric(df_clean['Amount'], errors='coerce')

# เติมค่า Amount ที่ขาดหายด้วย (Quantity * Unit_Price)
df_clean['Amount'] = df_clean['Amount'].fillna(
    df_clean['Quantity'] * df_clean['Unit_Price']
)

# แปลงวันที่ให้อยู่ในรูปแบบ YYYY-MM-DD
df_clean['Order_Date'] = pd.to_datetime(
    df_clean['Order_Date'], dayfirst=True, errors='coerce'
).dt.strftime('%Y-%m-%d')

print('Data Cleaning completed!')

Data Cleaning completed!


# **Phase 2A: TRANSFORM**

In [4]:
# สร้าง dim_customer
dim_customer = df_clean[['Customer_Name', 'Email']].drop_duplicates()

# สร้าง Surrogate Key (customer_id)
dim_customer = dim_customer.reset_index(drop=True)
dim_customer['customer_id'] = dim_customer.index + 1

# จัดเรียงคอลัมน์ให้ PK อยู่หน้าสุด
dim_customer = dim_customer[['customer_id', 'Customer_Name', 'Email']]

print('=== dim_customer ===')
dim_customer.head()

=== dim_customer ===


,customer_id,Customer_Name,Email
0,1,Emma Brown,emma.brown@email.com
1,2,Linda Park,linda.park@email.com
2,3,John Doe,john@email.com
3,4,Jane Smith,jane@email.com
4,5,Peter Kim,peter.kim@email.com


# **Phase 2B: TRANSFORM**

In [5]:
# นำ customer_id กลับไปใส่ใน fact table ผ่านการ Join
fact_sales = pd.merge(
    df_clean, dim_customer, on=['Customer_Name', 'Email'], how='left'
)

# ลบคอลัมน์ Text ทิ้ง เหลือไว้เพียง Foreign Key
fact_sales = fact_sales.drop(columns=['Customer_Name', 'Email'])

# จัดเรียงคอลัมน์ Fact Table ให้สวยงาม
fact_sales = fact_sales[[
    'Order_ID',
    'customer_id',
    'Order_Date',
    'Product',
    'Category',
    'Quantity',
    'Unit_Price',
    'Amount',
]]

print('=== fact_sales ===')
fact_sales.head()

=== fact_sales ===


,Order_ID,customer_id,Order_Date,Product,Category,Quantity,Unit_Price,Amount
0,ORD-0036,1,2026-04-24,Monitor 24 inch,Electronics,1,4131.0,4131.0
1,ORD-0076,2,NaN,Mechanical Keyboard,Electronics,5,1669.5,8347.5
2,ORD-0101,3,2026-04-11,Wireless Mouse,Electronics,3,405.0,1215.0
3,ORD-0059,4,2026-03-13,Wireless Mouse,Electronics,2,405.0,810.0
4,ORD-0038,5,NaN,Gel Pen Set,Stationery,5,85.5,427.5


# **Phase 3A: LOAD**

In [6]:
# 1. สร้าง Connection ไปที่ไฟล์ .db
conn = sqlite3.connect('warehouse.db')
cursor = conn.cursor()

# 2. สร้างตาราง Dimension พร้อมกำหนด Primary Key
cursor.execute('''
CREATE TABLE IF NOT EXISTS dim_customer (
    customer_id INTEGER PRIMARY KEY,
    Customer_Name TEXT,
    Email TEXT
)
''')

# สร้างตาราง Fact พร้อมกำหนด Foreign Key
cursor.execute('''
CREATE TABLE IF NOT EXISTS fact_sales (
    Order_ID TEXT,
    customer_id INTEGER,
    Order_Date TEXT,
    Product TEXT,
    Category TEXT,
    Quantity INTEGER,
    Unit_Price REAL,
    Amount REAL,
    FOREIGN KEY (customer_id) REFERENCES dim_customer (customer_id)
)
''')

conn.commit()
print('SQLite Warehouse Schema created successfully!')

SQLite Warehouse Schema created successfully!


# **Phase 3B: LOAD**

In [7]:
# เปิดใช้งานการตรวจสอบ Foreign Key ใน SQLite
cursor.execute('PRAGMA foreign_keys = ON;')

# สร้างตาราง Fact พร้อมผูก Foreign Key เข้ากับ Dimension Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS fact_sales (
    Order_ID TEXT,
    customer_id INTEGER,
    Order_Date TEXT,
    Product TEXT,
    Category TEXT,
    Quantity INTEGER,
    Unit_Price REAL,
    Amount REAL,
    FOREIGN KEY (customer_id) REFERENCES dim_customer (customer_id)
)
''')

conn.commit()
print('Star Schema Relationships set up successfully!')

Star Schema Relationships set up successfully!


# **Phase 3C: LOAD**

In [8]:
# โหลดข้อมูล Dimension
dim_customer.to_sql(
    'dim_customer', con=conn, if_exists='replace', index=False
)

# โหลดข้อมูล Fact
fact_sales.to_sql('fact_sales', con=conn, if_exists='replace', index=False)

print('ETL Pipeline ran successfully!')

ETL Pipeline ran successfully!


# **Verification**

In [9]:
conn = sqlite3.connect('warehouse.db')

# สั่ง JOIN ตาราง Fact และ Dimension เพื่อพิสูจน์การทำงานของ Star Schema
query = '''
SELECT
    c.Customer_Name,
    SUM(f.Amount) as Total_Spend
FROM fact_sales f
JOIN dim_customer c ON f.customer_id = c.customer_id
GROUP BY c.Customer_Name
ORDER BY Total_Spend DESC
LIMIT 3;
'''

df_result = pd.read_sql(query, conn)
print(df_result)

# ปิด Connection เมื่อเสร็จสิ้น
conn.close()

  Customer_Name  Total_Spend
0     Peter Kim     96283.25
1      Krit Som     88853.00
2    Alice Wong     81977.50
